In [1]:
import os
import zipfile
import json
import requests
import numpy as np
import pandas as pd

ARTIFACT_ZIP = "/content/premier_league_artifacts.zip"
ARTIFACT_DIR = "/content/premier_league_artifacts"

os.makedirs(ARTIFACT_DIR, exist_ok=True)

with zipfile.ZipFile(ARTIFACT_ZIP, "r") as zip_ref:
    zip_ref.extractall(ARTIFACT_DIR)

print("Extracted artifacts:")
for root, dirs, files in os.walk(ARTIFACT_DIR):
    for file in files:
        print(os.path.join(root, file))

Extracted artifacts:
/content/premier_league_artifacts/triplet_model.pt
/content/premier_league_artifacts/siamese_embeddings.npy
/content/premier_league_artifacts/autoencoder_model.pt
/content/premier_league_artifacts/triplet_embeddings.npy
/content/premier_league_artifacts/siamese_model.pt
/content/premier_league_artifacts/model_features.joblib
/content/premier_league_artifacts/feature_scaler.joblib
/content/premier_league_artifacts/player_metadata.csv
/content/premier_league_artifacts/dropped_features.joblib
/content/premier_league_artifacts/standard_embeddings.npy


In [2]:
for root, dirs, files in os.walk(ARTIFACT_DIR):
    if "player_metadata.csv" in files:
        REAL_ARTIFACT_DIR = root
        break
else:
    raise FileNotFoundError("Could not find player_metadata.csv")

print("Artifact directory:", REAL_ARTIFACT_DIR)

player_metadata = pd.read_csv(
    os.path.join(REAL_ARTIFACT_DIR, "player_metadata.csv")
)

standard_embeddings = np.load(
    os.path.join(REAL_ARTIFACT_DIR, "standard_embeddings.npy")
)

siamese_embeddings = np.load(
    os.path.join(REAL_ARTIFACT_DIR, "siamese_embeddings.npy")
)

triplet_embeddings = np.load(
    os.path.join(REAL_ARTIFACT_DIR, "triplet_embeddings.npy")
)

print("Players:", player_metadata.shape)
print("Standard embeddings:", standard_embeddings.shape)
print("Siamese embeddings:", siamese_embeddings.shape)
print("Triplet embeddings:", triplet_embeddings.shape)

Artifact directory: /content/premier_league_artifacts
Players: (397, 3)
Standard embeddings: (397, 41)
Siamese embeddings: (397, 16)
Triplet embeddings: (397, 8)


In [4]:
!pip install kaggle -q

In [5]:
import os
from google.colab import userdata

os.makedirs('/root/.kaggle', exist_ok=True)

KAGGLE_USERNAME = userdata.get('KAGGLE_USERNAME')
KAGGLE_KEY = userdata.get('KAGGLE_KEY')

with open('/root/.kaggle/kaggle.json', 'w') as f:
  f.write(f'{{"username":"{KAGGLE_USERNAME}","key":"{KAGGLE_KEY}"}}')

os.chmod('/root/.kaggle/kaggle.json', 0o600)

print("Kaggle API credentials configured.")

Kaggle API credentials configured.


In [6]:
kaggle_dataset_path = 'aesika/english-premier-league-player-stats-2425'

import os
dataset_dir = './english-premier-league-player-stats-2425'
os.makedirs(dataset_dir, exist_ok=True)

from kaggle.api.kaggle_api_extended import KaggleApi

api = KaggleApi()
api.authenticate()

print(f"Downloading dataset: {kaggle_dataset_path} to {dataset_dir}")
api.dataset_download_files(kaggle_dataset_path, path=dataset_dir, unzip=True)
print("Dataset downloaded and unzipped successfully!")

print("Files in the downloaded dataset directory:")
for root, dirs, files in os.walk(dataset_dir):
  for file in files:
    print(os.path.join(root, file))

Dataset URL: https://www.kaggle.com/datasets/aesika/english-premier-league-player-stats-2425
Dataset downloaded and unzipped successfully!
Files in the downloaded dataset directory:
./english-premier-league-player-stats-2425/epl_player_stats_24_25.csv


In [7]:
DATA_PATH = (
    "/content/english-premier-league-player-stats-2425/"
    "epl_player_stats_24_25.csv"
)

df = pd.read_csv(DATA_PATH)

percentage_cols = [
    "Conversion %",
    "Passes%",
    "Crosses %",
    "fThird Passes %",
    "gDuels %",
    "aDuels %",
    "Saves %"
]

for col in percentage_cols:
    df[col] = (
        df[col]
        .str.replace("%", "", regex=False)
        .astype(float)
    )

In [8]:
per90_cols = [
    "Goals",
    "Assists",
    "Shots",
    "Shots On Target",
    "Big Chances Missed",
    "Hit Woodwork",
    "Offsides",
    "Touches",
    "Passes",
    "Successful Passes",
    "Crosses",
    "Successful Crosses",
    "fThird Passes",
    "Successful fThird Passes",
    "Through Balls",
    "Carries",
    "Progressive Carries",
    "Carries Ended with Goal",
    "Carries Ended with Assist",
    "Carries Ended with Shot",
    "Carries Ended with Chance",
    "Possession Won",
    "Dispossessed",
    "Clean Sheets",
    "Clearances",
    "Interceptions",
    "Blocks",
    "Tackles",
    "Ground Duels",
    "gDuels Won",
    "Aerial Duels",
    "aDuels Won",
    "Goals Conceded",
    "xGoT Conceded",
    "Own Goals",
    "Fouls",
    "Yellow Cards",
    "Red Cards",
    "Saves",
    "Penalties Saved",
    "Clearances Off Line",
    "Punches",
    "High Claims",
    "Goals Prevented"
]

for col in per90_cols:
    df[f"{col}_per90"] = (
        df[col] / df["Minutes"]
    ) * 90

df_model = df[df["Minutes"] >= 450].copy()

print("Players after filtering:", len(df_model))

Players after filtering: 397


In [9]:
important_stats = [
    "Minutes",
    "Goals",
    "Assists",
    "Shots",
    "Shots On Target",
    "Passes%",
    "Conversion %",
    "Touches_per90",
    "Passes_per90",
    "Shots_per90",
    "Assists_per90",
    "Goals_per90",
    "Carries_per90",
    "Progressive Carries_per90",
    "Through Balls_per90",
    "Crosses_per90",
    "Tackles_per90",
    "Interceptions_per90",
    "Ground Duels_per90",
    "Aerial Duels_per90",
    "Saves_per90",
    "Clearances_per90",
    "Fouls_per90"
]

player_knowledge = df_model[
    ["Player Name", "Club", "Position"] + important_stats
].copy()

player_knowledge.head()

,Player Name,Club,Position,Minutes,Goals,Assists,Shots,Shots On Target,Passes%,Conversion %,...,Progressive Carries_per90,Through Balls_per90,Crosses_per90,Tackles_per90,Interceptions_per90,Ground Duels_per90,Aerial Duels_per90,Saves_per90,Clearances_per90,Fouls_per90
0,Ben White,Arsenal,DEF,1198,0,2,9,12,89.0,13.0,...,22.237062,0.300501,3.831386,1.502504,1.727880,17.353923,1.202003,0.000000,2.854758,0.751252
1,Bukayo Saka,Arsenal,MID,1735,6,10,67,2,87.0,25.0,...,3.579251,0.051873,0.051873,1.504323,0.778098,3.008646,2.334294,0.000000,0.311239,0.778098
2,David Raya,Arsenal,GKP,3420,0,0,0,0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.263158,0.763158,0.026316
3,Declan Rice,Arsenal,MID,2833,4,7,48,18,81.0,15.0,...,8.259795,0.349453,2.001412,1.683727,0.412990,10.864808,0.825980,0.000000,1.588422,0.667137
4,Ethan Nwaneri,Arsenal,MID,889,4,0,24,0,0.0,0.0,...,0.000000,0.000000,0.000000,1.113611,0.000000,0.000000,0.000000,0.000000,0.404949,0.911136


In [10]:
def player_to_document(row):

    return f"""
Player: {row['Player Name']}
Club: {row['Club']}
Position: {row['Position']}

Key Statistics:
Goals: {row['Goals']}
Assists: {row['Assists']}
Shots: {row['Shots']}
Shots On Target: {row['Shots On Target']}

Goals per 90: {row['Goals_per90']:.3f}
Assists per 90: {row['Assists_per90']:.3f}
Shots per 90: {row['Shots_per90']:.3f}
Touches per 90: {row['Touches_per90']:.3f}
Passes per 90: {row['Passes_per90']:.3f}
Carries per 90: {row['Carries_per90']:.3f}
Progressive Carries per 90: {row['Progressive Carries_per90']:.3f}
Through Balls per 90: {row['Through Balls_per90']:.3f}
Crosses per 90: {row['Crosses_per90']:.3f}

Pass Completion: {row['Passes%']}%
Conversion: {row['Conversion %']}%
Tackles per 90: {row['Tackles_per90']:.3f}
Interceptions per 90: {row['Interceptions_per90']:.3f}
Ground Duels per 90: {row['Ground Duels_per90']:.3f}
Aerial Duels per 90: {row['Aerial Duels_per90']:.3f}
"""

player_documents = {
    row["Player Name"]: player_to_document(row)
    for _, row in player_knowledge.iterrows()
}

print(player_documents["Bukayo Saka"])


Player: Bukayo Saka
Club: Arsenal
Position: MID

Key Statistics:
Goals: 6
Assists: 10
Shots: 67
Shots On Target: 2

Goals per 90: 0.311
Assists per 90: 0.519
Shots per 90: 3.476
Touches per 90: 56.749
Passes per 90: 33.354
Carries per 90: 8.663
Progressive Carries per 90: 3.579
Through Balls per 90: 0.052
Crosses per 90: 0.052

Pass Completion: 87.0%
Conversion: 25.0%
Tackles per 90: 1.504
Interceptions per 90: 0.778
Ground Duels per 90: 3.009
Aerial Duels per 90: 2.334



In [11]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrices = {
    "standard": cosine_similarity(standard_embeddings),
    "siamese": cosine_similarity(siamese_embeddings),
    "triplet": cosine_similarity(triplet_embeddings)
}

In [12]:
player_names = player_metadata["Player Name"].values


def retrieve_similar_players(
    player_name,
    top_k=5
):

    if player_name not in player_names:
        raise ValueError(
            f"{player_name} not found in player embeddings."
        )

    query_idx = np.where(
        player_names == player_name
    )[0][0]

    records = []

    for model_name, sim_matrix in similarity_matrices.items():

        scores = sim_matrix[query_idx].copy()
        scores[query_idx] = -1

        top_indices = np.argsort(scores)[::-1][:top_k]

        for rank, idx in enumerate(top_indices, start=1):

            records.append({
                "Model": model_name,
                "Player": player_names[idx],
                "Rank": rank,
                "Similarity": scores[idx]
            })

    results = pd.DataFrame(records)

    consensus = (
        results
        .groupby("Player")
        .agg(
            Models_Retrieved=("Model", "count"),
            Mean_Rank=("Rank", "mean"),
            Best_Rank=("Rank", "min")
        )
        .reset_index()
    )

    return consensus.sort_values(
        ["Models_Retrieved", "Mean_Rank"],
        ascending=[False, True]
    )

In [13]:
retrieve_similar_players(
    "Bukayo Saka",
    top_k=10
)

,Player,Models_Retrieved,Mean_Rank,Best_Rank
18,Matheus Cunha,2,3.0,2
12,Jacob Murphy,2,4.0,3
17,Leandro Trossard,2,5.5,1
13,Jarrod Bowen,2,6.5,4
10,Gonçalo Guedes,1,1.0,1
21,Mohammed Kudus,1,1.0,1
4,Cole Palmer,1,2.0,2
19,Mikel Merino,1,2.0,2
15,Julio Enciso,1,3.0,3
22,Morgan Rogers,1,3.0,3


In [14]:
from google.colab import userdata

ANAKIN_API_KEY = userdata.get("ANAKIN_API_KEY")

if not ANAKIN_API_KEY:
    raise ValueError(
        "ANAKIN_API_KEY not found in Colab Secrets."
    )

print("Anakin API key loaded.")

Anakin API key loaded.


In [15]:
def anakin_search(
    query,
    limit=5
):

    url = "https://api.anakin.io/v1/search"

    headers = {
        "X-API-Key": ANAKIN_API_KEY,
        "Content-Type": "application/json"
    }

    payload = {
        "prompt": query,
        "limit": limit
    }

    response = requests.post(
        url,
        headers=headers,
        json=payload,
        timeout=60
    )

    response.raise_for_status()

    return response.json()

In [16]:
search_result = anakin_search(
    "Bukayo Saka playing style tactical analysis Premier League",
    limit=5
)

print(search_result.keys())

dict_keys(['id', 'results'])


In [18]:
for result in search_result.get("results", []):

    print("\n" + "=" * 70)
    print("TITLE:", result.get("title"))
    print("URL:", result.get("url"))
    print("DATE:", result.get("date"))
    print("SNIPPET:", result.get("snippet"))


TITLE: Bukayo Saka Arsenal Forward, Profile & Stats | Premier League
URL: https://www.premierleague.com/en/players/223340/bukayo-saka/overview
DATE: 
SNIPPET: Bukayo Saka
Arsenal Arsenal • 7 Forward Follow ‌ Overview Video Matches Stats Career History
Next Match
‌
Teammates
‌ ‌ ‌ Arsenal
From the Clubs
See all * HAVERTZ, SAKA & ODEGAARD SECURE OUR OPENING-DAY WIN | HIGHLIGHTS | Arsenal vs Coventry (3-0) | EPL * Arteta's update on Guimaraes' injury after win over Coventry Club News * Tzolis on his flying start after dream Premier League return
Club App (IOS)
Club App (Android)
Matchday Hospitality
Club Shop
Listen live You will need to be signed in to your Premier League account to listen to live streams. Sign in or Register here Premier League The Premier League website uses essential cookies to make our website work. We would also like to use analytics cookies to improve your user experience. Non-essential cookies will be set only

TITLE: Analysing Bukayo Saka’s Premier League goals:

In [19]:
web_documents = []

for result in search_result.get("results", []):

    content = result.get("content", "")

    if not content:
        content = result.get("snippet", "")

    web_documents.append({
        "title": result.get("title"),
        "url": result.get("url"),
        "date": result.get("date"),
        "content": content
    })

len(web_documents)

5

In [20]:
print(search_result["results"][0].keys())

dict_keys(['date', 'last_updated', 'snippet', 'title', 'url'])


In [21]:
search_df = pd.DataFrame([
    {
        "title": r.get("title"),
        "url": r.get("url"),
        "date": r.get("date"),
        "last_updated": r.get("last_updated"),
        "snippet": r.get("snippet")
    }
    for r in search_result.get("results", [])
])

search_df

,title,url,date,last_updated,snippet
0,"Bukayo Saka Arsenal Forward, Profile & Stats |...",https://www.premierleague.com/en/players/22334...,,,Bukayo Saka\nArsenal Arsenal • 7 Forward Follo...
1,Analysing Bukayo Saka’s Premier League goals: ...,https://www.nytimes.com/athletic/5887698/2024/...,2024-11-06,2024-11-06,Saka doesn’t have a trademark finish. He’s onl...
2,Premier League 2022-23 review: players of the ...,https://www.theguardian.com/football/2023/may/...,2023-05-29,2023-05-29,"There’s not much on which football fans agree,..."
3,Bukayo Saka Stats This Season & Career Statist...,https://www.premierleague.com/en/players/22334...,,,Skip to main content\nListen live\nYou will ne...
4,The Breakdown: How Bukayo Saka has raised his ...,https://www.premierleague.com/en/video/4145637,,,Skip to main content


In [22]:
saka_url = search_df.loc[
    search_df["title"].str.contains(
        "Bukayo Saka Arsenal Forward",
        case=False,
        na=False
    ),
    "url"
].iloc[0]

saka_url

'https://www.premierleague.com/en/players/223340/bukayo-saka/overview'

In [23]:
def anakin_scrape(url, use_browser=False):

    endpoint = "https://api.anakin.io/v1/url-scraper/scrape"

    headers = {
        "X-API-Key": ANAKIN_API_KEY,
        "Content-Type": "application/json"
    }

    payload = {
        "url": url,
        "useBrowser": use_browser,
        "generateJson": False
    }

    response = requests.post(
        endpoint,
        headers=headers,
        json=payload,
        timeout=90
    )

    response.raise_for_status()

    return response.json()

In [24]:
saka_page = anakin_scrape(
    saka_url,
    use_browser=False
)

print(saka_page.keys())

dict_keys(['id', 'status', 'url', 'jobType', 'country', 'html', 'cleanedHtml', 'markdown', 'cached', 'createdAt', 'completedAt', 'durationMs'])


In [25]:
print(
    json.dumps(
        saka_page,
        indent=2
    )[:5000]
)

{
  "id": "d404c544-7427-43f4-a640-e8259eef664c",
  "status": "completed",
  "url": "https://www.premierleague.com/en/players/223340/bukayo-saka/overview",
  "jobType": "url_scraper",
  "country": "us",
  "html": "<!doctype html>\n<html lang=\"en\">\n\n    <link rel=\"alternate\" hreflang=\"x-default\" href=\"https://www.premierleague.com/en/players/223340/bukayo-saka/overview\" />\n        <link rel=\"alternate\" hreflang=\"en\" href=\"https://www.premierleague.com/en/players/223340/bukayo-saka/overview\" />\n        <link rel=\"alternate\" hreflang=\"es\" href=\"https://www.premierleague.com/es/players/223340/bukayo-saka/overview\" />\n        <link rel=\"alternate\" hreflang=\"ar\" href=\"https://www.premierleague.com/ar/players/223340/bukayo-saka/overview\" />\n\n\n<head>\n\t<meta charset=\"UTF-8\">\n\t<meta http-equiv=\"X-UA-Compatible\" content=\"IE=edge,chrome=1\">\n\t<meta name=\"viewport\" content=\"width=device-width, initial-scale=1\">\n\n\t<meta name=\"twitter:title\" conte

In [27]:
saka_page = anakin_scrape(
    saka_url,
    use_browser=False
)

print(saka_page.keys())

dict_keys(['id', 'status', 'url', 'jobType', 'country', 'html', 'cleanedHtml', 'markdown', 'cached', 'createdAt', 'completedAt', 'durationMs'])


In [29]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(
    saka_page["html"],
    "html.parser"
)

for tag in soup([
    "script",
    "style",
    "noscript",
    "nav",
    "footer"
]):
    tag.decompose()

saka_text = soup.get_text(
    separator="\n",
    strip=True
)

print(saka_text[:5000])

Bukayo Saka Arsenal Forward, Profile & Stats | Premier League
Skip to main content


In [30]:
lines = [
    line.strip()
    for line in saka_text.splitlines()
    if line.strip()
]

clean_saka_text = "\n".join(lines)

print(clean_saka_text[:5000])

Bukayo Saka Arsenal Forward, Profile & Stats | Premier League
Skip to main content


In [31]:
saka_web_document = {
    "title": search_df.loc[
        search_df["url"] == saka_url,
        "title"
    ].iloc[0],
    "url": saka_url,
    "source": "Premier League",
    "content": clean_saka_text
}

print(saka_web_document["title"])
print(saka_web_document["url"])
print(saka_web_document["content"][:3000])

Bukayo Saka Arsenal Forward, Profile & Stats | Premier League
https://www.premierleague.com/en/players/223340/bukayo-saka/overview
Bukayo Saka Arsenal Forward, Profile & Stats | Premier League
Skip to main content


In [32]:
web_documents = []

for _, row in search_df.iterrows():

    try:
        page = anakin_scrape(
            row["url"],
            use_browser=False
        )

        html = page.get("html", "")

        if not html:
            print("No HTML:", row["url"])
            continue

        soup = BeautifulSoup(
            html,
            "html.parser"
        )

        for tag in soup([
            "script",
            "style",
            "noscript",
            "nav",
            "footer"
        ]):
            tag.decompose()

        text = soup.get_text(
            separator="\n",
            strip=True
        )

        web_documents.append({
            "title": row["title"],
            "url": row["url"],
            "date": row["date"],
            "source": row["url"].split("/")[2],
            "content": text
        })

        print("Scraped:", row["title"])

    except Exception as e:
        print("Failed:", row["url"])
        print("Error:", e)

Scraped: Bukayo Saka Arsenal Forward, Profile & Stats | Premier League
Scraped: Analysing Bukayo Saka’s Premier League goals: Far-post curlers, slick combinations and rebounds - The Athletic
Scraped: Premier League 2022-23 review: players of the season | Premier League | The Guardian
Scraped: Bukayo Saka Stats This Season & Career Statistics | Premier League
Scraped: The Breakdown: How Bukayo Saka has raised his game to new heights


In [33]:
print("Documents scraped:", len(web_documents))

Documents scraped: 5


In [34]:
document_stats = pd.DataFrame([
    {
        "Title": doc["title"],
        "Source": doc["source"],
        "Characters": len(doc["content"]),
        "Words": len(doc["content"].split())
    }
    for doc in web_documents
])

document_stats.sort_values(
    "Words",
    ascending=False
)

,Title,Source,Characters,Words
1,Analysing Bukayo Saka’s Premier League goals: ...,www.nytimes.com,11740,1974
2,Premier League 2022-23 review: players of the ...,www.theguardian.com,7356,1229
4,The Breakdown: How Bukayo Saka has raised his ...,www.premierleague.com,86,16
3,Bukayo Saka Stats This Season & Career Statist...,www.premierleague.com,87,15
0,"Bukayo Saka Arsenal Forward, Profile & Stats |...",www.premierleague.com,82,14


In [35]:
substantive_web_docs = [
    doc for doc in web_documents
    if len(doc["content"].split()) >= 100
]

print("Substantive scraped documents:",
      len(substantive_web_docs))

for doc in substantive_web_docs:
    print(
        f"- {doc['title']} | "
        f"{len(doc['content'].split())} words"
    )

Substantive scraped documents: 2
- Analysing Bukayo Saka’s Premier League goals: Far-post curlers, slick combinations and rebounds - The Athletic | 1974 words
- Premier League 2022-23 review: players of the season | Premier League | The Guardian | 1229 words


In [36]:
snippet_documents = []

for _, row in search_df.iterrows():

    snippet = row["snippet"]

    if pd.notna(snippet) and len(str(snippet).split()) >= 10:

        snippet_documents.append({
            "title": row["title"],
            "url": row["url"],
            "date": row["date"],
            "source": row["url"].split("/")[2],
            "content": str(snippet)
        })

print("Snippet documents:", len(snippet_documents))

Snippet documents: 4


In [37]:
rag_documents = substantive_web_docs + snippet_documents

print("Total RAG documents:", len(rag_documents))

Total RAG documents: 6


In [38]:
def chunk_text(text, chunk_size=500, overlap=100):

    words = text.split()

    chunks = []

    start = 0

    while start < len(words):

        end = start + chunk_size

        chunk = " ".join(words[start:end])

        if chunk.strip():
            chunks.append(chunk)

        start += chunk_size - overlap

    return chunks

In [39]:
rag_chunks = []

for doc_id, doc in enumerate(rag_documents):

    chunks = chunk_text(
        doc["content"],
        chunk_size=500,
        overlap=100
    )

    for chunk_id, chunk in enumerate(chunks):

        rag_chunks.append({
            "doc_id": doc_id,
            "chunk_id": chunk_id,
            "title": doc["title"],
            "url": doc["url"],
            "source": doc["source"],
            "date": doc["date"],
            "text": chunk
        })

print("Total chunks:", len(rag_chunks))

Total chunks: 13


In [40]:
!pip -q install sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 71.6 MB/s eta 0:00:00


In [41]:
from sentence_transformers import SentenceTransformer

In [42]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded.


In [43]:
chunk_texts = [
    chunk["text"]
    for chunk in rag_chunks
]

rag_embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Embedding shape:", rag_embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding shape: (13, 384)


In [44]:
import faiss

embedding_dim = rag_embeddings.shape[1]

index = faiss.IndexFlatIP(
    embedding_dim
)

index.add(
    rag_embeddings.astype("float32")
)

print("Indexed chunks:", index.ntotal)

Indexed chunks: 13


In [45]:
def retrieve_web_context(
    query,
    top_k=5
):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(
        query_embedding,
        top_k
    )

    results = []

    for score, idx in zip(scores[0], indices[0]):

        chunk = rag_chunks[idx].copy()

        chunk["score"] = float(score)

        results.append(chunk)

    return results

In [46]:
results = retrieve_web_context(
    "How does Bukayo Saka score goals and where does he attack from?",
    top_k=5
)

for i, result in enumerate(results, start=1):

    print("\n" + "=" * 70)
    print(f"RESULT {i}")
    print("Score:", round(result["score"], 4))
    print("Title:", result["title"])
    print("Source:", result["source"])
    print("URL:", result["url"])
    print("\n", result["text"][:1000])


RESULT 1
Score: 0.6529
Title: Premier League 2022-23 review: players of the season | Premier League | The Guardian
Source: www.theguardian.com
URL: https://www.theguardian.com/football/2023/may/29/premier-league-2022-23-review-players-of-the-season

 There’s not much on which football fans agree, but the elemental and essential loveliness of Bukayo Saka is one. But make no mistake, behind the chasmic smile and sage youthfulness lives an absolute killer. His excellence at left-back, left-wing and right-wing evidence a rare footballing intelligence, and had Arsenal succeeded in signing Raphinha, he might have spent the season in midfield. Instead, though, he remained out wide, where his ability to draw defenders in, turn them with a flick of the buttocks and streak away – all in a single movement – are turning him into a world star, a lesson in studied dynamism. Rio Ferdinand once said that Paul Scholes’s passes “tell you where to go” and Saka is similar, except where he goes tells you 

In [47]:
def retrieve_diverse_web_context(
    query,
    top_k=5,
    max_per_source=2
):

    # Retrieve more candidates than needed
    candidates = retrieve_web_context(
        query,
        top_k=15
    )

    selected = []
    source_counts = {}

    for result in candidates:

        source = result["source"]

        if source_counts.get(source, 0) >= max_per_source:
            continue

        selected.append(result)
        source_counts[source] = (
            source_counts.get(source, 0) + 1
        )

        if len(selected) >= top_k:
            break

    return selected

In [48]:
results = retrieve_diverse_web_context(
    "How does Bukayo Saka play and what makes him effective?",
    top_k=5
)

for i, result in enumerate(results, 1):

    print("\n" + "=" * 70)
    print(f"RESULT {i}")
    print("Score:", round(result["score"], 4))
    print("Source:", result["source"])
    print("Title:", result["title"])
    print("\n", result["text"][:700])


RESULT 1
Score: 0.696
Source: www.theguardian.com
Title: Premier League 2022-23 review: players of the season | Premier League | The Guardian

 There’s not much on which football fans agree, but the elemental and essential loveliness of Bukayo Saka is one. But make no mistake, behind the chasmic smile and sage youthfulness lives an absolute killer. His excellence at left-back, left-wing and right-wing evidence a rare footballing intelligence, and had Arsenal succeeded in signing Raphinha, he might have spent the season in midfield. Instead, though, he remained out wide, where his ability to draw defenders in, turn them with a flick of the buttocks and streak away – all in a single movement – are turning him into a world star, a lesson in studied dynamism. Rio Ferdinand once said that Paul Scholes’s passes “tell you where to go” and 

RESULT 2
Score: 0.5229
Source: www.nytimes.com
Title: Analysing Bukayo Saka’s Premier League goals: Far-post curlers, slick combinations and rebounds - T

In [49]:
def get_player_stats(player_name):

    match = player_knowledge[
        player_knowledge["Player Name"] == player_name
    ]

    if len(match) == 0:
        return None

    row = match.iloc[0]

    return {
        "Player": row["Player Name"],
        "Club": row["Club"],
        "Position": row["Position"],
        "Goals": row["Goals"],
        "Assists": row["Assists"],
        "Goals_per90": row["Goals_per90"],
        "Assists_per90": row["Assists_per90"],
        "Shots_per90": row["Shots_per90"],
        "Touches_per90": row["Touches_per90"],
        "Passes_per90": row["Passes_per90"],
        "Carries_per90": row["Carries_per90"],
        "Progressive_Carries_per90": row["Progressive Carries_per90"],
        "Through_Balls_per90": row["Through Balls_per90"],
        "Crosses_per90": row["Crosses_per90"]
    }

In [51]:
def build_player_context(
    player_name,
    top_players=5,
    top_web=5
):

    # ML retrieval
    similar_players = retrieve_similar_players(
        player_name,
        top_k=top_players
    )

    # Web retrieval
    web_query = (
        f"{player_name} playing style tactical analysis "
        f"Premier League"
    )

    web_results = retrieve_diverse_web_context(
        web_query,
        top_k=top_web
    )

    # Query player statistics
    query_stats = get_player_stats(
        player_name
    )

    return {
        "query_player": query_stats,
        "similar_players": similar_players,
        "web_results": web_results
    }

In [52]:
context = build_player_context(
    "Bukayo Saka",
    top_players=5,
    top_web=5
)

print("QUERY PLAYER")
print(context["query_player"])

print("\nSIMILAR PLAYERS")
display(context["similar_players"])

print("\nWEB SOURCES")
for result in context["web_results"]:

    print(
        f"\n{result['title']} "
        f"({result['source']})"
    )

QUERY PLAYER
{'Player': 'Bukayo Saka', 'Club': 'Arsenal', 'Position': 'MID', 'Goals': np.int64(6), 'Assists': np.int64(10), 'Goals_per90': np.float64(0.3112391930835735), 'Assists_per90': np.float64(0.5187319884726225), 'Shots_per90': np.float64(3.4755043227665703), 'Touches_per90': np.float64(56.749279538904894), 'Passes_per90': np.float64(33.35446685878963), 'Carries_per90': np.float64(8.662824207492795), 'Progressive_Carries_per90': np.float64(3.579250720461095), 'Through_Balls_per90': np.float64(0.05187319884726225), 'Crosses_per90': np.float64(0.05187319884726225)}

SIMILAR PLAYERS


,Player,Models_Retrieved,Mean_Rank,Best_Rank
8,Matheus Cunha,2,3.0,2
4,Jacob Murphy,2,4.0,3
3,Gonçalo Guedes,1,1.0,1
7,Leandro Trossard,1,1.0,1
10,Mohammed Kudus,1,1.0,1
1,Cole Palmer,1,2.0,2
9,Mikel Merino,1,2.0,2
6,Julio Enciso,1,3.0,3
11,Morgan Rogers,1,3.0,3
2,Georginio Rutter,1,4.0,4



WEB SOURCES

Premier League 2022-23 review: players of the season | Premier League | The Guardian (www.theguardian.com)

Analysing Bukayo Saka’s Premier League goals: Far-post curlers, slick combinations and rebounds - The Athletic (www.nytimes.com)

Analysing Bukayo Saka’s Premier League goals: Far-post curlers, slick combinations and rebounds - The Athletic (www.nytimes.com)

Bukayo Saka Arsenal Forward, Profile & Stats | Premier League (www.premierleague.com)

Premier League 2022-23 review: players of the season | Premier League | The Guardian (www.theguardian.com)


In [53]:
!pip -q install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 4.3 MB/s eta 0:00:00


In [54]:
from google.colab import userdata
from groq import Groq

GROQ_API_KEY = userdata.get(
    "GROQ_API_KEY"
)

if not GROQ_API_KEY:
    raise ValueError(
        "GROQ_API_KEY not found in Colab Secrets."
    )

groq_client = Groq(
    api_key=GROQ_API_KEY
)

print("Groq client initialized.")

Groq client initialized.


In [55]:
def format_rag_context(context):

    player = context["query_player"]

    stats_text = f"""
QUERY PLAYER
Player: {player['Player']}
Club: {player['Club']}
Position: {player['Position']}

Goals: {player['Goals']}
Assists: {player['Assists']}
Goals per 90: {player['Goals_per90']:.3f}
Assists per 90: {player['Assists_per90']:.3f}
Shots per 90: {player['Shots_per90']:.3f}
Touches per 90: {player['Touches_per90']:.3f}
Passes per 90: {player['Passes_per90']:.3f}
Carries per 90: {player['Carries_per90']:.3f}
Progressive Carries per 90: {player['Progressive_Carries_per90']:.3f}
Through Balls per 90: {player['Through_Balls_per90']:.3f}
Crosses per 90: {player['Crosses_per90']:.3f}
"""

    similar_text = "\nSIMILAR PLAYERS\n"

    for _, row in context["similar_players"].head(5).iterrows():

        similar_text += (
            f"- {row['Player']} "
            f"(Models retrieved: "
            f"{row['Models_Retrieved']}, "
            f"Mean rank: "
            f"{row['Mean_Rank']:.2f})\n"
        )

    web_text = "\nWEB EVIDENCE\n"

    for i, result in enumerate(
        context["web_results"],
        1
    ):

        web_text += (
            f"\nSOURCE {i}\n"
            f"Title: {result['title']}\n"
            f"URL: {result['url']}\n"
            f"Evidence: {result['text']}\n"
        )

    return (
        stats_text +
        similar_text +
        web_text
    )

In [56]:
rag_context = format_rag_context(context)

system_prompt = """
You are a football analytics assistant.

Answer using ONLY the supplied evidence.

Rules:
1. Do not invent statistics.
2. Distinguish statistical similarity from tactical similarity.
3. If the evidence is insufficient, say so.
4. Explain why players are considered similar using the supplied statistics.
5. When using web evidence, cite the source title and URL.
6. Do not claim that model similarity proves two players are tactically identical.
"""

In [57]:
user_prompt = f"""
Answer this question:

"Who plays like Bukayo Saka, and why?"

Here is the retrieved evidence:

{rag_context}
"""

In [59]:
completion = groq_client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ],
    temperature=0.2,
    max_completion_tokens=1200
)

answer = completion.choices[0].message.content

print(answer)

**Statistically‑similar players (according to the supplied model)**  

| Player | How the model groups him with Saka |
|--------|------------------------------------|
| **Matheus Cunha** | Retrieved by the similarity model (mean rank 3.00). The model flagged Cunha because his per‑90 numbers for goals, assists, shots, touches, passes and carries are close to Saka’s (e.g., ≈0.3 goals/90, ≈0.5 assists/90, ≈3–4 shots/90, ≈55–60 touches/90, ≈30–35 passes/90, ≈8–9 carries/90). |
| **Jacob Murphy** | Retrieved by the model (mean rank 4.00). Murphy’s per‑90 output in the same categories sits near Saka’s, giving a comparable goal‑creation profile. |
| **Gonçalo Guedes** | Retrieved by the model (mean rank 1.00). Guedes shows a similar balance of scoring and creating – roughly the same goals‑per‑90 and assists‑per‑90 rates as Saka, plus comparable involvement in carries and progressive runs. |
| **Leandro Trossard** | Retrieved by the model (mean rank 1.00). Trossard’s per‑90 statistics (goals, 

In [60]:
from google.colab import userdata
from groq import Groq
import json
import numpy as np
import pandas as pd

GROQ_API_KEY = userdata.get("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found in Colab Secrets.")

client = Groq(api_key=GROQ_API_KEY)

MODEL = "openai/gpt-oss-120b"

print("Groq initialized.")

Groq initialized.


In [61]:
def ml_player_search(player_name, top_k=5):

    result = retrieve_similar_players(
        player_name,
        top_k=top_k
    )

    return result.to_dict(orient="records")

In [62]:
print(
    json.dumps(
        ml_player_search("Bukayo Saka", 5),
        indent=2
    )
)

[
  {
    "Player": "Matheus Cunha",
    "Models_Retrieved": 2,
    "Mean_Rank": 3.0,
    "Best_Rank": 2
  },
  {
    "Player": "Jacob Murphy",
    "Models_Retrieved": 2,
    "Mean_Rank": 4.0,
    "Best_Rank": 3
  },
  {
    "Player": "Gon\u00e7alo Guedes",
    "Models_Retrieved": 1,
    "Mean_Rank": 1.0,
    "Best_Rank": 1
  },
  {
    "Player": "Leandro Trossard",
    "Models_Retrieved": 1,
    "Mean_Rank": 1.0,
    "Best_Rank": 1
  },
  {
    "Player": "Mohammed Kudus",
    "Models_Retrieved": 1,
    "Mean_Rank": 1.0,
    "Best_Rank": 1
  },
  {
    "Player": "Cole Palmer",
    "Models_Retrieved": 1,
    "Mean_Rank": 2.0,
    "Best_Rank": 2
  },
  {
    "Player": "Mikel Merino",
    "Models_Retrieved": 1,
    "Mean_Rank": 2.0,
    "Best_Rank": 2
  },
  {
    "Player": "Julio Enciso",
    "Models_Retrieved": 1,
    "Mean_Rank": 3.0,
    "Best_Rank": 3
  },
  {
    "Player": "Morgan Rogers",
    "Models_Retrieved": 1,
    "Mean_Rank": 3.0,
    "Best_Rank": 3
  },
  {
    "Player": "Ge

In [66]:
def player_stats_tool(player_name):

    stats = get_player_stats(player_name)

    if stats is None:
        return {
            "error": f"Player '{player_name}' not found."
        }

    for key, value in stats.items():
        if isinstance(value, np.integer):
            stats[key] = int(value)
        elif isinstance(value, np.floating):
            stats[key] = float(value)

    return stats

In [67]:
print(
    json.dumps(
        player_stats_tool("Bukayo Saka"),
        indent=2
    )
)

{
  "Player": "Bukayo Saka",
  "Club": "Arsenal",
  "Position": "MID",
  "Goals": 6,
  "Assists": 10,
  "Goals_per90": 0.3112391930835735,
  "Assists_per90": 0.5187319884726225,
  "Shots_per90": 3.4755043227665703,
  "Touches_per90": 56.749279538904894,
  "Passes_per90": 33.35446685878963,
  "Carries_per90": 8.662824207492795,
  "Progressive_Carries_per90": 3.579250720461095,
  "Through_Balls_per90": 0.05187319884726225,
  "Crosses_per90": 0.05187319884726225
}


In [68]:
def search_football_web(query, limit=5):

    try:
        result = anakin_search(
            query,
            limit=limit
        )

        return [
            {
                "title": r.get("title"),
                "url": r.get("url"),
                "date": r.get("date"),
                "snippet": r.get("snippet")
            }
            for r in result.get("results", [])
        ]

    except Exception as e:

        return {
            "error": str(e)
        }

In [69]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "ml_player_search",
            "description": (
                "Find Premier League players statistically "
                "similar to a specified player using multiple "
                "machine-learning representations."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "player_name": {
                        "type": "string",
                        "description": "Player to find similar players for."
                    },
                    "top_k": {
                        "type": "integer",
                        "description": "Number of similar players to return.",
                        "minimum": 1,
                        "maximum": 10
                    }
                },
                "required": ["player_name"]
            }
        }
    },

    {
        "type": "function",
        "function": {
            "name": "player_stats_tool",
            "description": (
                "Get structured Premier League statistics "
                "for a player."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "player_name": {
                        "type": "string"
                    }
                },
                "required": ["player_name"]
            }
        }
    },

    {
        "type": "function",
        "function": {
            "name": "search_football_web",
            "description": (
                "Search the web for football news, tactical "
                "analysis, player role information and other "
                "relevant football context."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string"
                    },
                    "limit": {
                        "type": "integer",
                        "minimum": 1,
                        "maximum": 5
                    }
                },
                "required": ["query"]
            }
        }
    }
]

In [70]:
def execute_tool(tool_name, arguments):

    if tool_name == "ml_player_search":

        return ml_player_search(
            player_name=arguments["player_name"],
            top_k=arguments.get("top_k", 5)
        )

    elif tool_name == "player_stats_tool":

        return player_stats_tool(
            player_name=arguments["player_name"]
        )

    elif tool_name == "search_football_web":

        return search_football_web(
            query=arguments["query"],
            limit=arguments.get("limit", 5)
        )

    else:

        return {
            "error": f"Unknown tool: {tool_name}"
        }

In [71]:
SYSTEM_PROMPT = """
You are a Premier League football intelligence assistant.

You have access to three tools:

1. ml_player_search
   Finds statistically similar players using our ML models.

2. player_stats_tool
   Returns structured player statistics from our dataset.

3. search_football_web
   Searches the web for football context, tactical analysis,
   current information, and supporting evidence.

Rules:

- Use ML tools for statistical player similarity.
- Never invent player statistics.
- Use web search when tactical/contextual evidence is needed.
- Clearly distinguish statistical similarity from tactical similarity.
- Treat model similarity as evidence, not absolute truth.
- Use retrieved evidence when explaining your answer.
- Mention uncertainty when evidence is incomplete.
- Cite web sources using their title and URL.
"""

In [73]:
def run_agent(user_query, max_iterations=5):

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": user_query
        }
    ]

    for iteration in range(max_iterations):

        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
            temperature=0.2,
            max_completion_tokens=1500
        )

        message = response.choices[0].message

        messages.append(message)

        if not message.tool_calls:

            return message.content

        for tool_call in message.tool_calls:

            tool_name = tool_call.function.name

            arguments = json.loads(
                tool_call.function.arguments
            )

            print(
                f"Tool call: {tool_name}"
            )
            print(
                f"Arguments: {arguments}"
            )

            result = execute_tool(
                tool_name,
                arguments
            )

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": tool_name,
                "content": json.dumps(
                    result,
                    ensure_ascii=False
                )
            })

    return (
        "The agent reached the maximum number "
        "of tool iterations."
    )

In [74]:
answer = run_agent(
    "Who plays like Bukayo Saka, and why?"
)

print("\nFINAL ANSWER\n")
print(answer)

Tool call: ml_player_search
Arguments: {'player_name': 'Bukayo Saka', 'top_k': 5}
Tool call: player_stats_tool
Arguments: {'player_name': 'Bukayo Saka'}
Tool call: player_stats_tool
Arguments: {'player_name': 'Leandro Trossard'}
Tool call: player_stats_tool
Arguments: {'player_name': 'Mohammed Kudus'}
Tool call: player_stats_tool
Arguments: {'player_name': 'Cole Palmer'}

FINAL ANSWER

The agent reached the maximum number of tool iterations.


In [75]:
run_agent(
    "What are Bukayo Saka's goals, assists, shots per 90 and carries per 90?"
)

Tool call: player_stats_tool
Arguments: {'player_name': 'Bukayo Saka'}


'**Bukayo\u202fSaka – 2023/24 Premier League (Arsenal)**  \n\n| Metric | Value (per 90\u202fminutes) |\n|--------|------------------------|\n| Goals | **0.31**\u202fgoals per 90 |\n| Assists | **0.52**\u202fassists per 90 |\n| Shots | **3.48**\u202fshots per 90 |\n| Carries | **8.66**\u202fcarries per 90 |\n\n*Source:* Structured player statistics returned by the league data set (player_stats_tool).  \n\nThese per‑90 numbers are calculated from Saka’s total minutes played in the Premier League season, so they reflect his average contribution when he is on the pitch.'

In [76]:
run_agent(
    "Why is Bukayo Saka considered an effective right-sided attacker?"
)

Tool call: player_stats_tool
Arguments: {'player_name': 'Bukayo Saka'}
Tool call: ml_player_search
Arguments: {'player_name': 'Bukayo Saka', 'top_k': 5}
Tool call: search_football_web
Arguments: {'limit': 5, 'query': 'Bukayo Saka right-sided attacker analysis Arsenal tactics'}


'**Why Bukayo\u202fSaka is seen as an effective right‑sided attacker**\n\n| Aspect | What the data / analysis shows | Why it matters for a right‑winger |\n|--------|--------------------------------|-----------------------------------|\n| **Direct goal contribution** | 6\u202fgoals\u202f/\u202f90\u202fmin\u202f=\u202f0.31, 10\u202fassists\u202f/\u202f90\u202fmin\u202f=\u202f0.52 (≈\u202f0.83 goal‑involvements per 90)【player_stats_tool】 | A right‑winger is expected to add numbers both by scoring and by creating chances. Saka’s assist rate is among the highest for any Premier League wide‑player this season. |\n| **Progressive play** | 3.58\u202fprogressive\u202fcarries\u202f/\u202f90\u202fmin, 8.66\u202ftotal\u202fcarries\u202f/\u202f90\u202fmin【player_stats_tool】 | Progressive carries measure how often a player moves the ball into the opponent’s half. Saka repeatedly drives the ball forward on the flank, forcing defenders to step out of line. |\n| **Touch and passing volume** | 56.75\u20